## Notebook: Plot extreme events with all models
### Purpose : Load pre-trained models, run inference on held-out extreme days, and visualize spatial predictions.


In [ ]:
# --- Auto-reload for local package development ---
# Automatically reloads modified local modules without restarting the kernel

%load_ext autoreload
%autoreload 2

## Imports

In [ ]:
import os
import torch
import pandas as pd
import pickle
import numpy as np

from lightning_modelling.common_path import DATASET_PATH, MODELS_PATH
from lightning_modelling.deter_architecture import Unet, FullyConnectedNet_1d
from lightning_modelling.models import XGBoostModel, LogisticRegressionModel, GAMModel
from lightning_modelling.dataset import create_train_test
from lightning_modelling.plots import LightningPlotMultiModel

## Load data

In [ ]:
# Full range of available years in the dataset
ALL_YEARS = list(range(2008, 2024))

# Years reserved for evaluation only (Leave-One-Year-Out held-out set)
HELD_OUT_YEARS = [2008, 2015, 2023]

# Training years: all years except the held-out ones (LOYO strategy)
ALL_LOYO_YEARS = [year for year in ALL_YEARS if year not in HELD_OUT_YEARS]
TRAIN_YEARS = ALL_LOYO_YEARS
TEST_YEARS = HELD_OUT_YEARS

# Path to the pre-fitted scaler (fitted on the train years dataset)
SCALER_PATH = os.path.join(DATASET_PATH, "scaler", "scaler_full.pkl")

print("Creating the dataset..............")
# Only the test split is needed here; train/val outputs are discarded
_, _, TEST_DATASET = create_train_test(DATASET_PATH, TRAIN_YEARS, TEST_YEARS, scaler_path=SCALER_PATH)
TEST_DATASET.metadata_csv["year"] = pd.to_datetime(TEST_DATASET.metadata_csv["date"]).dt.year

# Load pre-computed extreme days (days with most lightning flashes observed)
extreme_days = pd.read_csv(DATASET_PATH / "extreme_days_top_0.05.csv")  

# Keep only extreme days that fall within the test years
all_extremes_metadata = TEST_DATASET.metadata_csv[TEST_DATASET.metadata_csv["date"].isin(extreme_days["date"])]
test_extremes = all_extremes_metadata[all_extremes_metadata["year"].isin(TEST_YEARS)]

## Load models

In [ ]:
# U-Net hyperparameters
CHANNELS = [16, 32, 64]
NUM_RESIDUAL_LAYERS = 2
RECALIBRATION = 'platt_scaling'

# Initialize unet
unet = Unet(
    channels=CHANNELS,
    num_residual_layers=NUM_RESIDUAL_LAYERS,
    name="unet",
    recalibration_method=RECALIBRATION,
)

unet.load_state_dict(torch.load(MODELS_PATH / 'unet.pth', map_location=torch.device('cpu')))
unet.eval()

xgb_name = "xgb"
xgb_model = pickle.load(open(MODELS_PATH / "xgb.pkl", "rb"))
xgb = XGBoostModel(model=xgb_model, name=xgb_name, remove_vars=None)


gam_name = "gam"
gam_model = pickle.load(open(MODELS_PATH / "gam.pkl", "rb"))
gam = GAMModel(model=gam_model, name=gam_name, remove_vars=None)


mlp_name = "mlp"
HIDDEN_DIMS = [16, 32, 16]

mlp = FullyConnectedNet_1d(
    name=mlp_name,
    save_path=None,
    hidden_dims=HIDDEN_DIMS,
    recalibration_method=RECALIBRATION,
    removed_features=[],
)

mlp.load_state_dict(torch.load(MODELS_PATH / "mlp.pth", map_location=torch.device('cpu')))
mlp.eval()

logreg_name = "logreg"
logreg_model = pickle.load(open(MODELS_PATH / "log_reg.pkl", "rb"))
logreg = LogisticRegressionModel(model=logreg_model, name=logreg_name, remove_vars=None)

models = [logreg, gam, xgb, mlp, unet]

## Plot extreme days

In [ ]:
# Preview the extreme day metadata
# Users can inspect this dataframe to then select specific dates in the next cell
test_extremes.head()

In [ ]:
# Select a subset of extreme dates for plotting
dates_to_plot = test_extremes["date"].tolist()[12:14]

for i, sample in enumerate(TEST_DATASET):
    date = test_extremes.iloc[i]["date"]
    
    # Skip samples not in the selected date window
    if date not in dates_to_plot:
        continue
    
    # Last channel is the observation; a lightning hour is defined as 
    # an hour where at least 2 flashes were observed in a grid cell
    obs = (sample[:, -1, :, :] >= 2).float()
    obs = obs.sum(axis=0).numpy()
    data = obs[np.newaxis, :, :]
    # --- Run inference for each model ---
    # All channels except the last (observation) are used as input features
    for model in models:
        preds = model(sample[:, :-1, :, :])
        if isinstance(preds, torch.Tensor):
            preds = preds.sum(axis=0).detach().numpy()
        data = np.concatenate([data, preds[np.newaxis, :, :]], axis=0)
        # data shape after loop: (1 + n_models, H, W)
    
    # --- Compute RMSE metrics per model --
    full_rmses = [] # RMSE across the full spatial domain
    conditionned_rmses = [] # RMSE restricted to grid cells with observed lightning (obs > 0)
    for j in range(1, data.shape[0]):
        full_rmses.append(
            np.sqrt(np.mean((data[0] - data[j])**2))
            )    
        conditionned_rmses.append(
            np.sqrt(np.mean((data[0][data[0] > 0] - data[j][data[0] > 0])**2))
            ) 
    
    # --- Define the colormap levels based on the max predicted/observed value ---
    max_hours = data.max()
    levels = np.linspace(0, int(np.ceil(max_hours)), int(round(max_hours)) + 1)
    
    # --- Plot all model outputs side by side for this event ---
    event_plot = LightningPlotMultiModel(
        lightning_hours=data,
        metadata_json=TEST_DATASET.metadata_json,
        title=f"{date} event",
        save_path=None,
        file_name=f"{date}.png",
        levels=levels,
        full_errors=full_rmses,
        conditionned_errors=conditionned_rmses
    )
    event_plot.show()
    # event_plot.save()  # Add a save_path and uncomment to save plots to disk